# StyleUp scoring proxy — Qwen3-VL edition

A notebook rewrite of the whole `proxy/` (the small Hono app in `proxy/src/`) that scores
one outfit photo per request and returns the exact JSON the Expo app already expects.
The only real change: **the model is Qwen3-VL running locally** instead of Gemini over HTTP,
so there is no API key, no per-photo cost, and nothing leaves your machine.

```
Expo app (phone)  ──POST /score {image: base64}──▶  this notebook (FastAPI on :8787)  ──▶  Qwen3-VL (local weights)
```

| Node proxy (`proxy/src/`) | This notebook | Cell |
| --- | --- | --- |
| `server.ts` — `serve(..., 8787)` | `uvicorn` in a background thread | *Start the server* |
| `app.ts` — Hono routes, CORS, validation | FastAPI routes, `CORSMiddleware`, same guards | *The proxy — FastAPI app* |
| `rubric.ts` — the prompt | `RUBRIC_PROMPT` + `REFERENCE_EXAMPLES` | *The rubric* |
| `score.ts` — Gemini client, hedging, normalization | `score_outfit()` against local Qwen3-VL | *Scoring* |
| `.env` — `GEMINI_*` / `PORT` | env vars `QWEN_*` / `PORT` | *Configuration* |

**Run order, top to bottom:** configuration → install → environment check → **download the model** →
rubric → helpers → load → scoring → server → smoke test. The download cell is the one that pulls
the weights to disk; everything after it works offline.

Latency note: a local 8B model does not match a hosted flash model, and the app's client only waits
90s (`src/lib/scoring.ts`). Pick the biggest Qwen3-VL your GPU can hold: ~6 GB VRAM → 2B, ~10 GB → 4B,
~18 GB → 8B, ~65 GB → 32B. CPU works too, just slowly (~1-3 min/photo).

## 1. Configuration

Every knob the Node proxy read from `.env` has the same name here, prefixed `QWEN_` where it used to
say `GEMINI_`. Override any of them with real environment variables, or just edit the defaults.

In [ ]:
"""
StyleUp scoring proxy — configuration
=====================================
    Node proxy (.env)          This notebook (env var -> global)
    ------------------------   ---------------------------------
    GEMINI_API_KEY             (none — weights run locally)
    GEMINI_MODEL               QWEN_MODEL
    GEMINI_FALLBACK_MODELS     QWEN_FALLBACK_MODELS
    GEMINI_TIMEOUT_MS          QWEN_TIMEOUT_MS
    GEMINI_HEDGE_MS            QWEN_HEDGE_MS
    PORT                       PORT
"""
import os
from pathlib import Path

# --- Which Qwen3-VL scores the outfits -------------------------------------
#   Dense : Qwen/Qwen3-VL-2B-Instruct · Qwen/Qwen3-VL-4B-Instruct
#           Qwen/Qwen3-VL-8B-Instruct · Qwen/Qwen3-VL-32B-Instruct
#   MoE   : Qwen/Qwen3-VL-30B-A3B-Instruct · Qwen/Qwen3-VL-235B-A22B-Instruct
#   FP8   : Qwen/Qwen3-VL-8B-Instruct-FP8  (Hopper/Ada + vLLM only)
MODEL_ID = os.environ.get("QWEN_MODEL", "Qwen/Qwen3-VL-8B-Instruct")

# Extra models to race the primary against when it stalls — the same hedge as the
# Gemini chain, comma-separated like GEMINI_FALLBACK_MODELS. Leave EMPTY on a single
# GPU: one device serialises generation anyway, so hedging only pays off when the
# fallbacks live on another machine or endpoint. If you do list models here, the
# download cell fetches them too.
FALLBACK_MODELS = [
    m.strip()
    for m in os.environ.get("QWEN_FALLBACK_MODELS", "").split(",")
    if m.strip()
]

# --- Server ----------------------------------------------------------------
PORT = int(os.environ.get("PORT", "8787"))
HOST = os.environ.get("HOST", "0.0.0.0")   # 0.0.0.0 so a phone on the same Wi-Fi can reach it

MAX_IMAGE_BYTES = 20_000_000               # base64 chars — the same guard as app.ts
ATTEMPT_TIMEOUT_MS = int(os.environ.get("QWEN_TIMEOUT_MS", "120000"))
HEDGE_DELAY_MS = int(os.environ.get("QWEN_HEDGE_MS", "20000"))

# --- Generation ------------------------------------------------------------
# Gemini was called with temperature 0.2 for a stable, comparable scale; keep that.
# (Qwen's own VL default is 0.7 — better for prose, worse for a rubric.)
MAX_NEW_TOKENS = 1024
TEMPERATURE = 0.2
TOP_P = 0.8
TOP_K = 20
REPETITION_PENALTY = 1.05

# --- Storage ---------------------------------------------------------------
MODELS_DIR = Path(os.environ.get("QWEN_MODELS_DIR") or (Path.cwd() / "models")).resolve()
HF_TOKEN = os.environ.get("HF_TOKEN") or None     # only for gated/private repos
LOCAL_FILES_ONLY = os.environ.get("QWEN_OFFLINE", "0") == "1"

print("model      :", MODEL_ID)
print("fallbacks  :", FALLBACK_MODELS or "(none)")
print("listening  :", f"http://{HOST}:{PORT}")
print("models dir :", MODELS_DIR)
print("timeouts   :", f"attempt {ATTEMPT_TIMEOUT_MS}ms, hedge {HEDGE_DELAY_MS}ms")

## 2. Install dependencies

The Node proxy needed Hono and nothing else; the notebook needs the vision stack.
`transformers >= 4.57` is the first release with native `Qwen3VLForConditionalGeneration`
support, so it is pinned here.

**Torch / CUDA:** if you are on Colab or a CUDA box, `torch` is already installed with the right
CUDA build — do not reinstall it. On a bare machine, install the CUDA build *first*:
`pip install torch --index-url https://download.pytorch.org/whl/cu124`, then run the cell below.

In [ ]:
%pip install -q --upgrade "transformers>=4.57.0" accelerate pillow "fastapi>=0.115" "uvicorn[standard]" huggingface_hub

In [ ]:
# Optional: flash-attention roughly halves VRAM for long visual contexts. It must
# match your torch/CUDA build, so it is not installed by default.
# %pip install -q flash-attn --no-build-isolation

## 3. Environment check

Decides the device, the weight dtype, and the attention backend once, so every later cell
just uses `DEVICE` / `DTYPE` / `ATTN_IMPL`.

In [ ]:
import importlib.util
import platform
import sys

import torch

print("python :", sys.version.split()[0], f"({platform.system()} {platform.machine()})")
print("torch  :", torch.__version__, "| cuda build:", torch.version.cuda)

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("gpu    :", props.name, f"| {props.total_memory / 1e9:.1f} GB VRAM | sm_{props.major}{props.minor}")
    DEVICE = torch.device("cuda:0")
    DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    DTYPE = torch.float16
    print("gpu    : Apple Metal (MPS)")
else:
    DEVICE = torch.device("cpu")
    DTYPE = torch.float32
    print("gpu    : none - running on CPU, expect 1-3 minutes per photo")

ATTN_IMPL = "flash_attention_2" if (DEVICE.type == "cuda" and importlib.util.find_spec("flash_attn")) else "sdpa"
print("dtype  :", DTYPE, "| attention:", ATTN_IMPL)

## 4. Download the model locally

This is the download script. It pulls every weight file for the primary model (and any
fallbacks) into `MODELS_DIR` using `huggingface_hub.snapshot_download`, which resumes
interrupted transfers and skips files that are already complete — so re-running the cell
costs nothing.

Layout: `models/Qwen__Qwen3-VL-8B-Instruct/`. Re-running with a bigger model later downloads
only the new weights; nothing is ever re-downloaded.

*(Private or gated repos only: export `HF_TOKEN=hf_...` before starting the kernel.)*

In [ ]:
from huggingface_hub import snapshot_download


def local_dir_for(model_id: str) -> Path:
    """models/Qwen__Qwen3-VL-8B-Instruct — flat, no nested org folder."""
    return MODELS_DIR / model_id.replace("/", "__")


def download_model(model_id: str, *, force: bool = False) -> str:
    """Fetch all weights for `model_id` into MODELS_DIR and return the local path."""
    target = local_dir_for(model_id)
    target.mkdir(parents=True, exist_ok=True)
    print(f"downloading {model_id}\n  -> {target}")

    path = snapshot_download(
        repo_id=model_id,
        local_dir=str(target),
        token=HF_TOKEN,
        max_workers=8,
        force_download=force,
        # Formats this notebook will never load: skip them and save tens of GB.
        ignore_patterns=["*.pth", "*.msgpack", "*.h5", "*.onnx*", "*.gguf", "original/*"],
    )

    total = sum(f.stat().st_size for f in Path(path).rglob("*") if f.is_file())
    print(f"  done - {total / 1e9:.2f} GB on disk\n")
    return path


MODEL_IDS = list(dict.fromkeys([MODEL_ID, *FALLBACK_MODELS]))
LOCAL_MODEL_PATHS = {model_id: download_model(model_id) for model_id in MODEL_IDS}

In [ ]:
# Verify the snapshot is complete — no network involved.
for model_id, path in LOCAL_MODEL_PATHS.items():
    root = Path(path)
    weights = sorted(root.glob("*.safetensors"))
    has_config = (root / "config.json").exists()
    has_processor = (root / "preprocessor_config.json").exists() or (root / "processor_config.json").exists()
    size = sum(f.stat().st_size for f in root.rglob("*") if f.is_file())
    print(f"{model_id}")
    print(f"  config.json            : {has_config}")
    print(f"  processor config       : {has_processor}")
    print(f"  weight shards (.safetensors): {len(weights)}")
    print(f"  total                  : {size / 1e9:.2f} GB")
    assert has_config and has_processor and weights, f"{model_id} looks incomplete — re-run the download cell"
print("\nAll snapshots look complete. The scoring cells can now run offline.")

## 5. The rubric

A byte-for-byte port of `proxy/src/rubric.ts`. **This file is the product** — the score is only
worth something if you agree with it, so calibrate `REFERENCE_EXAMPLES` the same way the README
describes: score 15-20 outfits you own and have opinions about, then write the clearest
disagreements into the list as anchors.

In [ ]:
# Anchors pin the scale to your taste. Add outfits with the score you believe they deserve:
#   "Navy unstructured blazer over a plain white tee, mid-wash straight jeans and leather
#    derbies: 86 — cohesive smart-casual palette, everything fits."
#   "Oversized graphic tee untucked over baggy cargos and chunky sneakers with a cap:
#    58 — pajama-adjacent; all volume, no anchor."
REFERENCE_EXAMPLES: list[str] = []

_references_section = (
    "\n## Reference outfits (anchors for the scale)\n"
    + "\n".join(f"- {example}" for example in REFERENCE_EXAMPLES)
    + "\n"
    if REFERENCE_EXAMPLES
    else ""
)

# What the wearer is dressing for. Same ids and wording as OCCASIONS in proxy/src/rubric.ts.
OCCASIONS: dict[str, str] = {
    "everyday": "a versatile everyday / casual-to-smart-casual context",
    "work": "a workday at an office with a business-casual dress code",
    "evening": "an evening out: a date, dinner, or drinks",
    "formal": "a formal event: a wedding, ceremony, or anything with a stated dress code",
    "active": "a workout or active day: gym, run, or sport",
}
DEFAULT_OCCASION = "everyday"

_RUBRIC_TEMPLATE = r"""You are a sharp, honest personal stylist. You score one outfit photo per request, the way a discerning friend who actually understands clothing would — specific, fair, never generic.

## Occasion

The wearer is dressing for __LENS__. Judge the whole outfit, and word every piece of feedback, for that occasion.

## Sub-scores (each an integer 0-100)

1. **fit_silhouette** — How garments fit and flatter: shoulder seams, jacket length, trouser break, tuck, hem, balance of volume (nothing swallow-you whole, nothing skin-tight where it shouldn't be), proportions between top and bottom.
2. **color_harmony** — Cohesion of the palette: a deliberate palette vs scattered colors, contrast that works against the wearer's skin/hair, at most one intentional pop of color, footwear/leather matching logic.
3. **occasion_appropriateness** — Does it read correctly for __LENS__? Nothing out of place, underdressed or overdressed for that lens.
4. **style_coherence** — One clear point of view (e.g. minimal, workwear, streetwear, preppy). Randomly assembled pieces score low even if each piece is nice alone.
5. **styling_details** — The finishing: tuck/cuff/roll, layering choices, accessories, footwear choice, how the pieces are worn rather than just chosen.

## Overall score (integer 0-100)

Judge holistically, not as a mechanical average. Use the full range:

- **90-100** — Exceptional. You would save this photo as a reference outfit.
- **80-89** — Genuinely good; only nitpicks.
- **70-79** — Solid but one change away from noticeably better.
- **60-69** — Several things working against each other.
- **50-59** — Mixed; a clash, bad proportion, or wrong fit dominates.
- **Below 50** — Misses on multiple dimensions.

Most decent outfits land 60-78. Reserve 90+ for outfits that are genuinely inspiring. Do not inflate, and do not be cruel.

## Feedback rules

- **strengths**: 2-3 items. Each must name an actual visible garment or detail ("the olive chinos break cleanly over the boots"). No generic praise ("nice colors") unless the palette is specifically described.
- **improvements**: 2-3 items. Each must be a swap-level action naming the garment and a concrete replacement or adjustment, e.g. "Swap the untucked boxy tee for a slightly tapered tee and tuck it to define the waist." Prioritize the single change with the highest payoff first.
- **summary**: one sentence, direct verdict.

## Output

Return ONLY a JSON object, no markdown fences, no commentary, with exactly these keys, written in exactly this order. The order matters: describe what you see before you judge it, and settle the overall score last, once the sub-scores and feedback are written, so the number follows from the evidence.
{
  "outfit_visible": <true if the photo shows clothing being worn, or an outfit laid out, clearly enough to judge; false for anything else (no person, no clothes, a screenshot, an object, too dark or too cropped to tell)>,
  "observations": "<2-3 sentences, description only, no judgment: each visible garment, its color, its fit, and how it is worn>",
  "subscores": {
    "fit_silhouette": <int 0-100>,
    "color_harmony": <int 0-100>,
    "occasion_appropriateness": <int 0-100>,
    "style_coherence": <int 0-100>,
    "styling_details": <int 0-100>
  },
  "strengths": [<string>, ...],
  "improvements": [<string>, ...],
  "summary": "<string>",
  "score": <int 0-100>
}

If "outfit_visible" is false, do not judge anything: say what the photo shows in "observations", set every score to 0, and leave "strengths" and "improvements" empty.
""" + _references_section


def build_rubric_prompt(occasion: str = DEFAULT_OCCASION) -> str:
    """Port of buildRubricPrompt(): the rubric, judged through one occasion's lens."""
    return _RUBRIC_TEMPLATE.replace("__LENS__", OCCASIONS[occasion])


RUBRIC_PROMPT = build_rubric_prompt()

print(f"rubric: {len(RUBRIC_PROMPT)} chars | {len(REFERENCE_EXAMPLES)} reference anchor(s)")

## 6. Response helpers

The defensive half of `score.ts`, unchanged in spirit: never trust the model to return exactly
what it was asked for. Models also sometimes wrap JSON in fences or prose, so the extractor
falls back to the outermost `{...}` span before giving up.

In [ ]:
import json
import re

SUBSCORE_KEYS = (
    "fit_silhouette",
    "color_harmony",
    "occasion_appropriateness",
    "style_coherence",
    "styling_details",
)


def require_score(value, field: str) -> int:
    """Port of requireScore: a 0-100 number the model actually gave, or an error.
    A missing or junk value is rejected, never replaced — an invented number would be
    indistinguishable from a real one once it is on screen and in history."""
    try:
        if isinstance(value, bool) or not isinstance(value, (int, float, str)):
            raise ValueError
        number = float(value)
        if number != number or number in (float("inf"), float("-inf")):
            raise ValueError
        number = round(number)
        if not 0 <= number <= 100:
            raise ValueError
    except ValueError:
        raise ValueError(f'"{field}" is missing or not a 0-100 number') from None
    return int(number)


def to_string_array(value) -> list[str]:
    if not isinstance(value, list):
        return []
    return [s for s in (str(v).strip() for v in value) if s]


def extract_json(text: str):
    """The model *should* emit pure JSON, but sometimes wraps it in fences or prose."""
    trimmed = (text or "").strip()
    try:
        return json.loads(trimmed)
    except json.JSONDecodeError:
        pass
    fenced = re.search(r"```(?:json)?\s*([\s\S]*?)```", trimmed)
    if fenced:
        try:
            return json.loads(fenced.group(1))
        except json.JSONDecodeError:
            pass
    start, end = trimmed.find("{"), trimmed.rfind("}")
    if start >= 0 and end > start:
        return json.loads(trimmed[start : end + 1])
    raise ValueError("No JSON object found in model response")


def normalize_score(raw) -> dict:
    """Shape the model's payload into the exact response the app validates in scoring.ts."""
    if not isinstance(raw, dict):
        raise ValueError("Score payload is not an object")

    subs_raw = raw.get("subscores") if isinstance(raw.get("subscores"), dict) else {}
    subscores = {key: require_score(subs_raw.get(key), f"subscores.{key}") for key in SUBSCORE_KEYS}

    strengths = to_string_array(raw.get("strengths"))[:3]
    improvements = to_string_array(raw.get("improvements"))[:3]
    summary = raw.get("summary").strip() if isinstance(raw.get("summary"), str) else ""

    if not strengths and not improvements and not summary:
        raise ValueError("Score payload has no usable content")

    return {
        "score": require_score(raw.get("score"), "score"),
        "subscores": subscores,
        "strengths": strengths,
        "improvements": improvements,
        "summary": summary,
    }


# Quick self-check of the tricky paths.
assert require_score("87.6", "score") == 88
for _junk in (None, "", "great", 500, -1, True):
    try:
        require_score(_junk, "score")
    except ValueError:
        continue
    raise AssertionError(f"require_score accepted {_junk!r}")
assert extract_json('```json\n{"score": 71}\n```')["score"] == 71
assert extract_json('Sure! {"score": 64} hope that helps')["score"] == 64
_full_subs = {key: 70 for key in SUBSCORE_KEYS} | {"color_harmony": 90}
assert normalize_score({"score": 80, "subscores": _full_subs, "summary": "ok"})["subscores"]["color_harmony"] == 90
try:
    normalize_score({"score": 80, "subscores": {"color_harmony": 90}, "summary": "ok"})
except ValueError:
    pass
else:
    raise AssertionError("normalize_score accepted a payload with missing subscores")
print("helpers OK")

## 7. Load the model

Weights load lazily, on the first request, and are cached per model id. `device_map="auto"`
shards across whatever GPUs you have. A single `threading.Lock` serialises generation — one GPU
can only do one forward pass at a time, and this also means a hedged second attempt queues up
rather than thrashing VRAM.

In [ ]:
import threading

from transformers import AutoProcessor, Qwen3VLForConditionalGeneration

_GPU_LOCK = threading.Lock()      # one GPU -> one generation at a time
_LOAD_LOCK = threading.Lock()
_LOADED: dict[str, tuple] = {}    # model_id -> (model, processor)


def model_device(model) -> torch.device:
    return next(model.parameters()).device


def is_model_loaded(model_id: str) -> bool:
    return model_id in _LOADED


def get_model(model_id: str):
    """Load (and cache) `model_id`, preferring the local snapshot over the Hub."""
    with _LOAD_LOCK:
        if model_id in _LOADED:
            return _LOADED[model_id]

        source = LOCAL_MODEL_PATHS.get(model_id, model_id)
        local_only = model_id in LOCAL_MODEL_PATHS or LOCAL_FILES_ONLY
        print(f"[model] loading {model_id}\n        from {source}")

        processor = AutoProcessor.from_pretrained(source, token=HF_TOKEN, local_files_only=local_only)
        model = Qwen3VLForConditionalGeneration.from_pretrained(
            source,
            dtype=DTYPE,
            device_map="auto",
            attn_implementation=ATTN_IMPL,
            token=HF_TOKEN,
            local_files_only=local_only,
        )
        model.eval()
        _LOADED[model_id] = (model, processor)
        print(f"[model] {model_id} ready on {model_device(model)} ({DTYPE})")
        return _LOADED[model_id]


def unload_model(model_id: str | None = None) -> None:
    """Drop weights and free VRAM — handy before downloading a bigger model."""
    import gc

    with _LOAD_LOCK:
        for key in list(_LOADED) if model_id is None else [model_id]:
            _LOADED.pop(key, None)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("[model] unloaded", model_id or "(all)")


# Pre-load now so the first phone request is not the one that pays the load cost.
# (Comment out to keep startup fast; the first /score will load it anyway.)
_ = get_model(MODEL_ID)

## 8. Scoring

The rest of `score.ts`:

- **`callModel` → `_generate_blocking`** — build the same `{text, image}` chat turn, decode the
  base64 photo to a PIL image, generate, and normalize.
- **`AbortController` → `threading.Event` + `StoppingCriteria`** — the losers of the hedge race
  stop generating mid-decode instead of burning GPU time to completion.
- **`GeminiError` → `QwenError`** — the same `(message, status, retryable)` contract, so a bad
  photo is never retried on another model but a crash or timeout is.

In [ ]:
import asyncio
import base64
import io
import time

from PIL import Image
from transformers import StoppingCriteria, StoppingCriteriaList


class QwenError(RuntimeError):
    """Port of GeminiError: HTTP-ish status + whether another model could do better."""

    def __init__(self, message: str, status: int | None = None, retryable: bool = True):
        super().__init__(message)
        self.name = "QwenError"
        self.status = status
        self.retryable = retryable


class _Cancelled(Exception):
    """Raised inside the worker thread when the race no longer needs this answer."""


class _CancelStopping(StoppingCriteria):
    """Local stand-in for controller.abort(): quit decoding the moment the attempt is
    abandoned (another model won) or the attempt deadline passes."""

    def __init__(self, cancel_event: threading.Event, deadline: float):
        self.cancel_event = cancel_event
        self.deadline = deadline

    def __call__(self, input_ids, scores, **kwargs) -> bool:
        return self.cancel_event.is_set() or time.monotonic() >= self.deadline


def decode_image(image_base64: str, mime_type: str = "image/jpeg") -> Image.Image:
    """base64 (bare or `data:` URL) -> RGB PIL image. Validated here, like the Node
    proxy validated the string, but so a broken upload fails fast and is not retried."""
    payload = image_base64.split(",", 1)[1] if image_base64.startswith("data:") else image_base64
    pad = len(payload) % 4
    if pad:
        payload += "=" * (4 - pad)
    try:
        raw = base64.b64decode(payload)
    except Exception as err:
        raise QwenError(f"image is not valid base64: {err}", 400, retryable=False) from err
    try:
        image = Image.open(io.BytesIO(raw))
        image.load()
    except Exception as err:
        raise QwenError(f"image could not be decoded: {err}", 400, retryable=False) from err
    if mime_type not in ("image/jpeg", "image/jpg", "image/png", "image/webp", "image/heic"):
        print(f"[score] note: unexpected mimeType {mime_type!r}; decoding by content instead")
    return image.convert("RGB")


@torch.inference_mode()
def _generate_blocking(model_id: str, image: Image.Image, prompt: str, cancel_event: threading.Event, deadline: float) -> str:
    """One Qwen3-VL generation. Runs in a worker thread; never called concurrently."""
    if cancel_event.is_set():
        raise _Cancelled()

    with _GPU_LOCK:                      # one GPU, one forward pass
        if cancel_event.is_set():
            raise _Cancelled()
        try:
            model, processor = get_model(model_id)
            messages = [
                {
                    "role": "user",
                    "content": [
                        {"type": "image", "image": image},   # PIL images are accepted directly
                        {"type": "text", "text": prompt},
                    ],
                }
            ]
            inputs = processor.apply_chat_template(
                messages,
                tokenize=True,
                add_generation_prompt=True,
                return_dict=True,
                return_tensors="pt",
            ).to(model_device(model))

            generated = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=TEMPERATURE > 0,
                temperature=TEMPERATURE,
                top_p=TOP_P,
                top_k=TOP_K,
                repetition_penalty=REPETITION_PENALTY,
                stopping_criteria=StoppingCriteriaList([_CancelStopping(cancel_event, deadline)]),
            )
            trimmed = generated[:, inputs["input_ids"].shape[1] :]
            text = processor.batch_decode(trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
        except torch.cuda.OutOfMemoryError as err:
            torch.cuda.empty_cache()
            raise QwenError(f"{model_id} ran out of VRAM: {err}", 500, retryable=True) from err
        except _Cancelled:
            raise
        except Exception as err:
            raise QwenError(f"{model_id} failed: {err}", None, retryable=True) from err

    if cancel_event.is_set():
        raise _Cancelled()
    return text


async def _attempt(model_id: str, image: Image.Image, prompt: str, cancel_event: threading.Event, timeout_ms: int) -> dict:
    """callModel(): run one model, bounded by an attempt timeout, abortable."""
    loop = asyncio.get_running_loop()
    deadline = time.monotonic() + timeout_ms / 1000
    future = loop.run_in_executor(None, _generate_blocking, model_id, image, prompt, cancel_event, deadline)
    try:
        text = await asyncio.wait_for(asyncio.shield(future), timeout_ms / 1000)
    except asyncio.TimeoutError as err:
        cancel_event.set()               # stop the worker, not just the await
        raise QwenError(f"{model_id} timed out after {timeout_ms}ms", None, retryable=True) from err
    except asyncio.CancelledError:
        cancel_event.set()
        raise
    except _Cancelled as err:
        raise QwenError(f"{model_id} was cancelled", None, retryable=True) from err

    if not text or not text.strip():
        # No safety filters locally — an empty answer means the model or the sampler broke.
        raise QwenError(f"{model_id} → empty response", None, retryable=False)
    parsed = extract_json(text)
    # Checked before the numbers: a "0" for a photo of a sofa is not a score.
    if isinstance(parsed, dict) and parsed.get("outfit_visible") is False:
        raise QwenError(
            "Couldn't find an outfit in this photo. Retake it with the whole outfit in frame, head to toe.",
            422,
            retryable=False,
        )
    return normalize_score(parsed)

## 9. The hedge

Same policy as the Gemini chain, and worth reading before you add fallbacks: the primary starts,
and if it has not answered within `HEDGE_DELAY_MS` — or fails outright — the next model is launched
alongside it. The first valid score wins and the losers are aborted. A failure is a signal, so the
next model starts immediately rather than waiting out the hedge timer.

On one GPU the second attempt simply queues behind `_GPU_LOCK`; hedging there buys availability
(a model that crashes or OOMs does not fail the request), not latency. Across separate endpoints it
buys both.

In [ ]:
async def score_outfit(image_base64: str, mime_type: str = "image/jpeg", occasion: str = DEFAULT_OCCASION) -> dict:
    """Port of scoreOutfit(): decode once, then hedge across the model chain."""
    prompt = build_rubric_prompt(occasion)
    image = decode_image(image_base64, mime_type)
    models = list(dict.fromkeys([MODEL_ID, *FALLBACK_MODELS]))
    started_at = time.monotonic()

    cancel_events: list[threading.Event] = []
    pending: set[asyncio.Task] = set()
    failures: list[str] = []

    def launch(index: int) -> asyncio.Task:
        """Start model `index`. Named after its model, so the winner is identifiable."""
        event = threading.Event()
        cancel_events.append(event)
        coro = _attempt(models[index], image, prompt, event, ATTEMPT_TIMEOUT_MS)
        return asyncio.create_task(coro, name=f"attempt-{models[index]}")

    def settle(task: asyncio.Task):
        """The response if this attempt succeeded; None if it failed and others keep going."""
        error = task.exception()
        if error is None:
            model_name = task.get_name().removeprefix("attempt-")
            ms = int((time.monotonic() - started_at) * 1000)
            print(f"[score] {model_name} answered in {ms}ms" + ("" if model_name == MODEL_ID else " (fallback)"))
            return {**task.result(), "model": model_name, "fallback": model_name != MODEL_ID}

        show = str(error)
        failures.append(show)
        print(f"[score] {show}")
        if isinstance(error, QwenError) and not error.retryable:
            # A rejected photo or malformed request won't fare better elsewhere.
            raise error
        return None

    try:
        for index in range(len(models)):
            pending.add(launch(index))
            # Don't wait for this one: give it HEDGE_DELAY_MS, then start the next alongside it.
            done, _ = await asyncio.wait(pending, timeout=HEDGE_DELAY_MS / 1000)
            for task in done:
                pending.discard(task)
                result = settle(task)
                if result is not None:
                    return result

        while pending:                    # every model launched; wait for whichever is left
            done, pending = await asyncio.wait(pending, return_when=asyncio.FIRST_COMPLETED)
            for task in done:
                result = settle(task)
                if result is not None:
                    return result

        last = failures[-1] if failures else "no models configured"
        raise RuntimeError(f"Qwen3-VL could not score this outfit — try again in a moment. ({last})")
    finally:
        for event in cancel_events:
            event.set()                   # stops in-flight generations mid-decode
        for task in pending:
            task.cancel()

## 10. The proxy — FastAPI app

A straight port of `app.ts`: permissive CORS like Hono's `cors()`, `GET /health`, and
`POST /score` with the identical validation order and error strings, so the app in `src/lib/scoring.ts`
cannot tell the difference.

| Check | Node proxy | Here |
| --- | --- | --- |
| Body is JSON | `c.req.json()` throw → 400 `Request body must be JSON` | same |
| `image` present | 400 `"image" (base64 string) is required` | same |
| `image` size | >20 MB chars → 413 `Image too large` | same |
| `mimeType` | defaults to `image/jpeg` | same |
| Scoring failure | 502 `{error}` | same |

In [ ]:
import sys

from fastapi import FastAPI, Request
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse

app = FastAPI(title="StyleUp scoring proxy (Qwen3-VL)", version="1.0.0")

# Hono's cors() default: every origin, every method, every header.
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=False,
    allow_methods=["*"],
    allow_headers=["*"],
)


@app.get("/health")
async def health():
    return {
        "ok": True,
        "model": MODEL_ID,
        "fallbacks": FALLBACK_MODELS,
        "backend": "local-transformers",
        "device": str(DEVICE),
        "dtype": str(DTYPE).replace("torch.", ""),
        "loaded": is_model_loaded(MODEL_ID),
        "localPath": str(LOCAL_MODEL_PATHS.get(MODEL_ID, "")),
    }


@app.post("/score")
async def score(request: Request):
    try:
        body = await request.json()
    except Exception:
        return JSONResponse({"error": "Request body must be JSON"}, status_code=400)

    image = body.get("image") if isinstance(body, dict) else None
    if not isinstance(image, str) or len(image) == 0:
        return JSONResponse({"error": '"image" (base64 string) is required'}, status_code=400)
    if len(image) > MAX_IMAGE_BYTES:
        return JSONResponse({"error": "Image too large"}, status_code=413)

    mime_type = body.get("mimeType") if isinstance(body.get("mimeType"), str) else "image/jpeg"

    # Absent = the default lens (older clients); present but unknown = a client bug.
    occasion = body.get("occasion", DEFAULT_OCCASION)
    if not isinstance(occasion, str) or occasion not in OCCASIONS:
        return JSONResponse(
            {"error": '"occasion" must be one of: ' + ", ".join(OCCASIONS)}, status_code=400
        )

    try:
        return JSONResponse(await score_outfit(image, mime_type, occasion))
    except QwenError as err:
        message = str(err)
        print(f"[score] {message}", file=sys.stderr)
        status = err.status if err.status and 400 <= err.status < 600 else 502
        return JSONResponse({"error": message}, status_code=status)
    except Exception as err:                      # noqa: BLE001 — same catch-all as app.ts
        message = str(err) or "Unknown error"
        print(f"[score] {message}", file=sys.stderr)
        return JSONResponse({"error": message}, status_code=502)

## 11. Start the server

`uvicorn` runs in a daemon thread so the notebook stays interactive. Prints the URLs to paste into
the app (**Settings → Scoring server → Save & test**):

- **iOS simulator** — `http://localhost:8787`
- **Android emulator** — `http://10.0.2.2:8787`
- **Physical phone** — `http://<LAN-IP>:8787`, same Wi-Fi, computer firewall permitting

In [ ]:
import socket
import threading
import time

import uvicorn

_SERVER: dict = {}


def lan_ip() -> str:
    """Best-effort LAN address — what a phone on the same Wi-Fi should point at."""
    sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    try:
        sock.connect(("8.8.8.8", 80))          # no packets sent, just a route lookup
        return sock.getsockname()[0]
    except OSError:
        return "127.0.0.1"
    finally:
        sock.close()


def start_server(port: int = PORT, host: str = HOST, wait: float = 15.0):
    if _SERVER.get("server") is not None and not _SERVER["server"].should_exit:
        print("already running")
        return _SERVER["server"]

    config = uvicorn.Config(app, host=host, port=port, log_level="info", access_log=False)
    server = uvicorn.Server(config)
    thread = threading.Thread(target=server.run, name="styleup-proxy", daemon=True)
    thread.start()
    _SERVER.update(server=server, thread=thread, port=port)

    deadline = time.monotonic() + wait
    while not server.started and time.monotonic() < deadline:
        time.sleep(0.1)
    if not server.started:
        raise RuntimeError("uvicorn did not come up — check the port is free")

    ip = lan_ip()
    print(f"StyleUp scoring proxy listening on http://{host}:{port}")
    print(f"  health check : http://localhost:{port}/health")
    print(f"  iOS sim      : http://localhost:{port}")
    print(f"  Android emu  : http://10.0.2.2:{port}")
    print(f"  physical phone: http://{ip}:{port}")
    return server


def stop_server():
    server = _SERVER.get("server")
    if server is None:
        return
    server.should_exit = True
    _SERVER["thread"].join(timeout=10)
    _SERVER.clear()
    print("server stopped")


start_server()

## 12. Smoke test

Builds a crude synthetic "outfit" photo (no camera needed), posts it to `/score` exactly the way
the phone does, and prints the result. This runs a real generation, so give it the time your GPU
needs.

In [ ]:
import json as _json
import urllib.request

from IPython.display import display
from PIL import Image as PILImage
from PIL import ImageDraw


def make_test_photo(path: str = "smoke_test_outfit.jpg") -> str:
    """A blocky stand-in for a head-to-toe photo: navy top, grey chinos, brown shoes."""
    img = PILImage.new("RGB", (768, 1024), (243, 241, 236))
    draw = ImageDraw.Draw(img)
    draw.ellipse((304, 110, 464, 270), fill=(206, 172, 146))            # head
    draw.rounded_rectangle((285, 300, 483, 610), 26, fill=(38, 55, 82))  # navy tee
    draw.rectangle((247, 340, 285, 550), fill=(38, 55, 82))              # sleeves
    draw.rectangle((483, 340, 521, 550), fill=(38, 55, 82))
    draw.rounded_rectangle((296, 610, 472, 900), 14, fill=(150, 148, 143))  # grey chinos
    draw.rounded_rectangle((300, 900, 370, 950), 8, fill=(92, 62, 40))      # shoes
    draw.rounded_rectangle((400, 900, 470, 950), 8, fill=(92, 62, 40))
    img.save(path, quality=90)
    return path


def score_file(path: str, base_url: str | None = None) -> dict:
    """Score an image file through the running proxy — the calibration workhorse."""
    if base_url is None:
        port = _SERVER.get("port", PORT) if "_SERVER" in globals() else PORT
        base_url = f"http://localhost:{port}"
    base = base_url.rstrip("/")
    image_b64 = base64.b64encode(Path(path).read_bytes()).decode()
    request = urllib.request.Request(
        f"{base}/score",
        data=_json.dumps({"image": image_b64, "mimeType": "image/jpeg"}).encode(),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(request, timeout=120) as response:
        return _json.loads(response.read())


photo = make_test_photo()
started = time.monotonic()
result = score_file(photo)
elapsed = time.monotonic() - started

print(f"\n{result['score']}/100  |  {result['model']}  |  {elapsed:.1f}s")
for key, value in result["subscores"].items():
    print(f"  {key:<26} {value}")
print("\nstrengths:")
for line in result["strengths"]:
    print("  +", line)
print("\nimprovements:")
for line in result["improvements"]:
    print("  ->", line)
print("\nsummary:", result["summary"])
display(PILImage.open(photo))

In [ ]:
# Health, exactly as the app's "Save & test" button sees it. Self-contained, so it works
# even if you skipped the smoke test.
import json as _health_json
import urllib.request as _urllib_request

_port = _SERVER.get("port", PORT) if "_SERVER" in globals() else PORT
with _urllib_request.urlopen(f"http://localhost:{_port}/health", timeout=10) as _response:
    print(_health_json.dumps(_health_json.loads(_response.read()), indent=2))

### Scoring a folder (calibration)

The README's acceptance test is agreement within ±5 points on 16 of 20 real outfits. Drop photos in
a folder and run this to collect your side-by-side numbers, then move the disagreements into
`REFERENCE_EXAMPLES` in cell 5 and re-run.

In [ ]:
def score_folder(folder: str, pattern: str = "*") -> list[dict]:
    """Score every image in `folder` and return rows you can argue with."""
    rows = []
    for path in sorted(Path(folder).glob(pattern)):
        if path.suffix.lower() not in {".jpg", ".jpeg", ".png", ".webp", ".heic"}:
            continue
        try:
            result = score_file(str(path))
            rows.append({
                "photo": path.name,
                "score": result["score"],
                **result["subscores"],
                "summary": result["summary"],
            })
            print(f"{result['score']:>3}  {path.name}  {result['summary']}")
        except Exception as err:                  # noqa: BLE001 — keep scoring the rest
            print(f"  !  {path.name}: {err}")
    return rows


# rows = score_folder("outfits")           # <- your folder of already-owned outfits
# import pandas as pd; pd.DataFrame(rows)  # <- then write down where you disagree by >5

## 14. Teardown

Stop the server and free VRAM before loading a different model or shutting the kernel down.

In [ ]:
# stop_server()
# unload_model()          # frees the weights from VRAM
# unload_model("Qwen/Qwen3-VL-4B-Instruct")

## Appendix — parity notes

**Behaviour kept identical to the Node proxy** — response shape (`score`, `subscores`, `strengths`,
`improvements`, `summary`, `model`, `fallback`), the `occasion` request field, status codes and error strings, the 20 MB base64 guard, the
`image/jpeg` default, permissive CORS, subscore order, and the ±5-point calibration workflow with
`REFERENCE_EXAMPLES` anchors.

**Deliberate differences**

| Topic | Node proxy | Notebook |
| --- | --- | --- |
| Model | Gemini flash chain, `GEMINI_API_KEY` | Qwen3-VL weights, downloaded locally |
| Key | required, lives only in the proxy | none — nothing leaves the machine |
| Timeout | 20s per attempt (a hosted model, heavily contested) | 120s per attempt (local weights are slower) |
| Hedge | wins 34-43s → ~5s under load | availability only on a single GPU; real latency win across endpoints |
| `model` field | e.g. `gemini-3.8-flash` | the Hugging Face repo id |
| Cloud deploy | deploys unchanged to Cloudflare Workers | notebook stays on the machine with the GPU |
| Overload error | `Gemini is overloaded right now — try again in a moment.` | `Qwen3-VL could not score this outfit — try again in a moment.` |
| Repeatability | temperature 0, optional median of `GEMINI_SAMPLES` runs | temperature 0.2, single run |
| Unusable score | one retry on the same model, then the next | straight to the next model |
| Access token | `PROXY_TOKEN` guards `/score` | none: keep the notebook on your LAN |
| Blocked photos | `promptFeedback.blockReason` → no retry | no local filter; decode failures → 400, no retry |

**Gotchas**

- A 90s limit in `src/lib/scoring.ts` is the real ceiling. If an 8B model cannot finish an outfit in
  time on your GPU, step down to 4B or 2B (`QWEN_MODEL`) rather than raising the timeout in the app.
- Scores are not comparable across different Qwen3-VL sizes. During calibration pin one model and an
  empty `QWEN_FALLBACK_MODELS`, exactly as the README advises for the Gemini chain.
- `models/` is gitignored; the weights never enter the repo.
- Keep the port free — if a previous run left a server behind, `stop_server()` before `start_server()`.